<a href="https://colab.research.google.com/github/MGentieu/dl_project/blob/main/starters/cv-project-starter/cv-project/notebooks/CV_Caltech-101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connexion à github

Pour github : à exécuter dans le terminal de colab

```bash
git clone https://github.com/MGentieu/dl_project.git

# CV Starter — Colab/Kaggle Quickstart

> Author : Badr TAJINI

**Academic year:** 2025–2026  
**School:** ECE  
**Course:** Machine Learning & Deep Learning 2

---


Welcome! This notebook is intentionally beginner-friendly. Follow the steps exactly and you will confirm that the starter project works on a free GPU runtime.

### Before you run anything
1. **Open the notebook in Google Colab or Kaggle.**
2. **Change the hardware accelerator to GPU (T4 preferred).**
   * Colab: `Runtime` → `Change runtime type` → Hardware accelerator `GPU` → Save.
   * Kaggle: `Settings` (gear icon) → Turn on `Accelerator` → Choose `T4 x1`.
3. Once the GPU is enabled, run the cells **from top to bottom**. Every code cell has comments explaining what it does.

The notebook will: (a) check that a GPU is available, (b) install dependencies, (c) run a quick smoke test that loads CIFAR-10 and performs one training step, and (d) show you how to launch the full training/evaluation commands when you are ready for longer experiments.

## 0. Project files

- Option A: `git clone <your_repo_url> cv-project`
- Option B: Upload the `cv-project` folder via the left sidebar (ensure the root is `/content/cv-project`).

Run the cell below afterwards; it will raise a helpful error if the folder is missing.

### Quick checklist before running code

- **Colab:** Go to `Runtime → Change runtime type`, pick `GPU`, and click **Save**.
- **Kaggle:** Open the gear icon in the top-right, enable **Accelerator**, and choose `T4 x1`.
- Wait for the runtime to restart (Colab shows `Connected` again).
- Then run every cell in order. If you see an error, stop, read the message, and re-run the cell after fixing the issue.

Once the smoke test succeeds you can run the full training and evaluation commands shown at the end of the notebook.

### How to run a cell
- Click the little ▶️ button on the left of a cell, or press **Shift + Enter** (Colab) / **Ctrl + Enter** (Kaggle).
- Wait for the cell to finish (a number like `[1]` appears once it is done).
- If a cell shows an error, read the message, fix the issue, and re-run that same cell before moving forward.


### Step 0 — Confirm the GPU is ready
Run the next cell. You should see a table with GPU details (name + memory).
If you get the message `nvidia-smi unavailable`, the runtime is still on CPU—go back to the checklist above and switch it to GPU, then rerun the cell.


In [1]:
!nvidia-smi || echo "nvidia-smi unavailable (CPU runtime)"

Sat Nov 29 17:49:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

On vérifie ensuite qu'on est bien en train d'utiliser un GPU. Si c'est le cas, la cellule en-dessous affiche **cuda** :

In [2]:
# Install required libraries
!pip -q install torch torchvision torchmetrics==1.4.0 matplotlib tqdm ultralytics datasets scikit-learn

# Set seeds for reproducibility
import torch, random, numpy as np
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Set device to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [7]:
# Install project dependencies listed in requirements.txt
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pretrainedmodels: filename=pretrainedmodels-0.7.4-py3-none-any.whl size=60945 sha256=356a19f0a2cfafa270d494276494cc2ed11cd9b7db76c1d9522093ae12bc1939
  Stored in directory: /root/.cache/pip/wheels/4c/01/56/40a48f75dbdfe167a0cb70d3b48913369a00ec5c4e9fed5f2b
Successfully built pretrainedmodels


### Step 1 — Point the notebook at the project folder
This cell makes sure Colab/Kaggle is looking at the `cv-project` directory.
If it raises a `FileNotFoundError`, you likely uploaded the folder to a different place. Use the file browser on the left to confirm the path, fix it, and rerun the cell.


In [4]:
import os
import sys
from pathlib import Path

# The user has provided the explicit path to the project root.
PROJECT_ROOT = Path("/content/dl_project/starters/cv-project-starter/cv-project")

print(f"Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT")

# Validate structure
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Missing src/ directory at {PROJECT_ROOT}")

# Setup Python path
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")


Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT
Project root: /content/dl_project/starters/cv-project-starter/cv-project
Working directory: /content/dl_project/starters/cv-project-starter/cv-project


### Installation du dataset

In [5]:
import torchvision as tv
tf_train = tv.transforms.Compose([tv.transforms.Resize(224), tv.transforms.RandomHorizontalFlip(), tv.transforms.ToTensor()])
tf_val   = tv.transforms.Compose([tv.transforms.Resize(224), tv.transforms.ToTensor()])
train = tv.datasets.Caltech101(root="./data", download=True, transform=tf_train)
# Create a small validation split:
val_size = int(0.1*len(train)); train, val = torch.utils.data.random_split(train, [len(train)-val_size, val_size])
val.dataset.transform = tf_val
print(len(train), len(val))

7810 867


### Step 2 — Install the project requirements
This command reads `requirements.txt` and installs exactly the same packages you would get locally.
Expect a lot of text output; that is normal. If installation fails, run the cell again before moving forward.


In [8]:
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
import joblib
import cv2
import os
import time
import random
import pretrainedmodels
import numpy as np

from imutils import paths
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Load torch...!!!
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Load torchvision ...!!!
from torchvision import transforms

'''SEED Everything'''
def seed_everything(SEED=42):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True # keep True if all the input have same size.
SEED=42
seed_everything(SEED=SEED)
'''SEED Everything'''

'SEED Everything'

### Step 3 — Run the smoke test
This quick check downloads CIFAR-10 (if needed), runs one mini-batch through the model, and saves `outputs/smoke_metrics.json`.
You should see a short JSON output such as `{"loss": ..., "batch_size": 64, ...}`.
If you hit a download or network error, wait a few seconds and re-run the cell; Colab sometimes needs a second try.


In [6]:
import yaml
cfg = yaml.safe_load(open("configs/cv_caltech101_fast.yaml"))
print(cfg["data"]["dataset"])


caltech101


In [7]:
from src.utils import load_yaml
cfg = load_yaml("configs/cv_caltech101_fast.yaml")
print(cfg["data"]["dataset"])

caltech101


In [ ]:
image_paths = list(paths.list_images('./101_ObjectCategories'))

data = []
labels = []
for img_path in tqdm(image_paths):
    label = img_path.split(os.path.sep)[-2]
    if label == "BACKGROUND_Google":
        continue
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    data.append(img)
    labels.append(label)

data = np.array(data)
labels = np.array(labels)

In [9]:
from src import smoke_check

smoke_path = smoke_check.run_smoke("configs/cv_caltech101_fast.yaml")
print(smoke_path.read_text())

TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 212, in collate
    collate(samples, collate_fn_map=collate_fn_map)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 240, in collate
    raise T

### Mise à jour du fichier `src/smoke_check.py`

La cellule suivante va réécrire le contenu de `src/smoke_check.py` pour inclure la transformation `ToTensor()` et un modèle simple nécessaire au bon fonctionnement du test de fumée.


In [19]:
%%writefile src/smoke_check_caltech.py
"""
Lightweight sanity check for the CV starter.

Loads CIFAR-10 using the configured transforms, runs a single forward/backward
pass, and stores metrics under outputs/smoke_metrics.json. Helpful for CPU-only
validation before longer GPU runs (e.g., on Colab/Kaggle T4).
"""

from pathlib import Path
import torch
import torch.nn as nn

from utils import load_yaml, set_seed, get_device, save_json
from data import build_dataloaders
from model import build_model


def run_smoke(cfg_path) -> Path:
    cfg = load_yaml(cfg_path)
    set_seed(cfg["seed"])
    device = get_device()

    train_loader, _, num_classes, _ = build_dataloaders(cfg)
    model = build_model(cfg, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])

    model.train()
    x, y = next(iter(train_loader))
    x, y = x.to(device), y.to(device)

    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()

    out_dir = Path(cfg["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "smoke_metrics.json"
    save_json(
        {
            "loss": float(loss.item()),
            "batch_size": int(x.size(0)),
            "num_classes": num_classes,
            "device": str(device),
        },
        out_path,
    )
    return out_path


if __name__ == "__main__":
    import sys
    cfg_path = sys.argv[1] if len(sys.argv) > 1 else "configs/cv_cifar10_fast.yaml"
    path = run_smoke(cfg_path)
    print(f"Smoke check succeeded. Metrics written to {path}.")



Writing src/smoke_check_caltech.py


> **Why run the smoke test?**

> The smoke test is a safety check before long training. It loads CIFAR‑10, runs a single forward/backward pass, and writes outputs/smoke_metrics.json. If this succeeds, you know:

> - the dataset can be downloaded/read,
> - the model compiles and runs on your GPU,
> - dependencies are installed correctly.
>
> In short, if it fails, fix the error (missing folder, bad install, no GPU) before investing time in a full training run :
>
> *After !python src/evaluate.py ... (Step 4 below)*

## 1. Review smoke-test output
- Confirm the previous cell printed a JSON block (loss, batch size, device).
- You should now see `outputs/smoke_metrics.json` in the file browser on the left.
- Need only a quick check? You can stop here. Ready for real training? Continue to Section 2 below.
- If anything failed, read the error message carefully, fix the issue, and re-run the smoke cell before moving on.


## 2. Full training run (optional)
Only run these cells when you want the complete experiment. On the first run PyTorch will also download CIFAR-10, so the first epoch may start slowly.

**Before running:**
1. Open `configs/cv_cifar10.yaml` (Menu → File → Open …) if you want to change hyperparameters.
2. Ensure the runtime still shows a GPU connection.
3. Close other browser tabs or notebooks to avoid memory pressure.


In [ ]:
# Train the model using the main configuration file. Expect visible progress bars.
# The first time you run this it will also download CIFAR-10, so the progress bar
# might pause around 0% while data is fetched.
!python src/train.py --config configs/cv_cifar10.yaml

In [ ]:
# Evaluate the best checkpoint produced during training.
# This will print accuracy and write the metrics/plots listed below.
!python src/evaluate.py --config configs/cv_cifar10.yaml --ckpt outputs/best.pt

### Step 4 — What should I see now?
- `outputs/best.pt`: saved checkpoint.
- `outputs/log.csv`: training history (loss/accuracy per epoch).
- `outputs/eval.json`, `per_class_metrics.csv`, `confusion_matrix.png`, `leaderboard.png`: evaluation artefacts.

If any of these files are missing, scroll up for errors in the training/evaluation cells.


## 3. Mirror this workflow for other tracks

1. Duplicate this notebook and rename it (e.g., `00_nlp_quickstart.ipynb`).
2. Copy the corresponding starter folder into your Colab/Kaggle workspace (`nlp-project`, `od-project`, `ts-project`).
3. Update the install cell so it matches that starter's `requirements.txt`.
4. Replace `from src import smoke_check` with the helper module provided in the new starter (each repo ships with one).
5. Point the train/eval commands at the new `configs/*.yaml` file.
6. Optionally tweak the markdown text so instructions mention the right dataset and metrics.

By following the same structure—GPU check → install → smoke test → full run—students can master all four tracks with a consistent workflow.

# On affiche à présent les résultats obtenus :

1. Affichage des métriques globales du modèle :

In [ ]:
import json

with open("outputs/metrics.json", "r") as f:
    metrics = json.load(f)

metrics

2. Affichage de la matrice de confusion

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("outputs/confusion_matrix.png")
plt.imshow(img)
plt.axis('off')
plt.show()


3. Affichage de leaderboard

In [ ]:
img = Image.open("outputs/leaderboard.png")
plt.imshow(img)
plt.axis('off')
plt.show()


4. Affichage des résultats en fonction des époques

In [ ]:
import pandas as pd

df_logs = pd.read_csv("outputs/log.csv")
df_logs.head()


5. Affichage des métriques du meilleur modèle par classe

In [ ]:
df_per_class = pd.read_csv("outputs/per_class_metrics.csv")
df_per_class


6. Évolution de la fonction de perte à l'entraînement et à la validation en fonction des époques

In [ ]:
plt.plot(df_logs['epoch'], df_logs['train_loss'], label='Train loss')
plt.plot(df_logs['epoch'], df_logs['val_loss'], label='Validation loss')
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


7. Affichage des métriques globales pour le meilleur modèle obtenu.

In [ ]:
with open("outputs/eval.json") as f:
    eval_results = json.load(f)

eval_results


# Task
Verify the `TypeError` encountered during the smoke test for the `Caltech101` dataset by re-running the test after applying the `ToTensor()` transformation fix in `src/smoke_check.py`. If the `TypeError` is resolved, integrate this fix into `src/data.py`'s `build_dataloaders` function to ensure images are correctly transformed into tensors.

## Apply ToTensor() fix to smoke_check.py

### Subtask:
Update `src/smoke_check.py` with the provided `ToTensor()` transformation fix to address the `TypeError` encountered during the smoke test.


**Reasoning**:
The subtask requires overwriting `src/smoke_check.py` with the provided corrected `smoke_check_caltech.py` content to fix the `TypeError`. This cell will achieve that by using the `%%writefile` magic command.



**Reasoning**:
The `src/smoke_check.py` file has been updated. Now, rerun the smoke test to verify that the `TypeError` has been resolved and that the smoke test runs successfully.



**Reasoning**:
The previous smoke test failed with a `RuntimeError` indicating a mismatch in image channel dimensions (1 channel vs. 3 channels) during normalization. This suggests that some images might be loaded as grayscale by `Caltech101` but the normalization is expecting 3 channels. To fix this, I need to explicitly convert images to RGB format before applying `ToTensor()` and `Normalize` transformations.



**Reasoning**:
The `src/smoke_check.py` file has been updated to handle potential grayscale images by explicitly converting them to RGB. Now, the smoke test needs to be re-executed to confirm that this fix resolves the `RuntimeError` related to channel mismatch during normalization and that the smoke test runs successfully.



**Reasoning**:
The previous attempts to fix the RuntimeError related to channel mismatch during normalization were unsuccessful. The error indicates that despite `convert('RGB')`, some images are still resulting in single-channel tensors (`[1, 224, 224]`). To address this robustly, I will add a `transforms.Lambda` function *after* `transforms.ToTensor()` to explicitly check the number of channels and duplicate the single channel into three if necessary, ensuring a `[3, 224, 224]` tensor for normalization. I will also set `num_workers=0` in `DataLoader` to rule out any multiprocessing-related issues during this debugging phase.



**Reasoning**:
The `src/smoke_check.py` file has been updated with the explicit channel handling and `num_workers=0`. Now, the smoke test needs to be re-executed to confirm that this fix resolves the `RuntimeError` and that the smoke test runs successfully.



**Reasoning**:
The previous attempts to fix the `RuntimeError` related to channel mismatch during normalization were unsuccessful. The error persisted, indicating that the `Normalize` transform was still receiving single-channel tensors (`[1, 224, 224]`). To robustly ensure all images have three channels before normalization, I will modify the transformation pipeline to explicitly use `transforms.Grayscale(num_output_channels=3)` on the PIL image before converting it to a tensor. This ensures a 3-channel input to `ToTensor()`, guaranteeing a 3-channel tensor for `Normalize`.



**Reasoning**:
The `src/smoke_check.py` file has been updated with `transforms.Grayscale(num_output_channels=3)`. Now, the smoke test needs to be re-executed to confirm that this fix resolves the `RuntimeError` related to channel mismatch during normalization and that the smoke test runs successfully.



**Reasoning**:
The subtask specifically instructs to overwrite `src/smoke_check.py` with the content of cell `11332527` to apply the `ToTensor()` transformation fix, as part of verifying the `TypeError` resolution. This step is about adhering to that explicit instruction.

